# Olist ecommerce Analytics

In [ ]:
import pandas as pd

file_path = (r"C:\Users\hp\Downloads\Telegram Desktop\Ecommerce_Analytics\E commerce\Anasta Project 2 Ecommerce.xlsx")

excel = pd.ExcelFile(file_path)

print(excel.sheet_names[:9])

In [ ]:
customers = pd.read_excel(file_path, sheet_name='Customers')

orders = pd.read_excel(file_path, sheet_name='Orders')

geolocation = pd.read_excel(file_path, sheet_name='Geolocation')

order_items = pd.read_excel(file_path, sheet_name='Order_Items')

product_category = pd.read_excel(file_path, sheet_name='Product_cat_name')

sellers = pd.read_excel(file_path, sheet_name='Sellers')

products = pd.read_excel(file_path, sheet_name='Products')

reviews = pd.read_excel(file_path, sheet_name='Order_Reviews')

payments = pd.read_excel(file_path, sheet_name='Order_Payments')

print("All 9 datasets loaded successfully!")

In [ ]:
datasets = {
    'Customers': customers,
    'Orders': orders,
    'Geolocation': geolocation,
    'Order Items': order_items,
    'Product Category': product_category,
    'Sellers': sellers,
    'Products': products,
    'Reviews': reviews,
    'Payments': payments
}

for name, df in datasets.items():
    print(name, ":", df.shape)

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print(df.columns.tolist())

In [ ]:
for name, df in datasets.items():
    print("\n==========", name, "==========")
    for column in df.columns:
        print(column)

In [ ]:
# Convert Orders date columns
order_date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in order_date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')


# Convert Order Items date column
order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date'], errors='coerce'
)


# Convert Review date columns
reviews['review_creation_date'] = pd.to_datetime(
    reviews['review_creation_date'], errors='coerce'
)

reviews['review_answer_timestamp'] = pd.to_datetime(
    reviews['review_answer_timestamp'], errors='coerce'
)


print("Date conversion completed!")
print("\nOrders date types:")
print(orders[order_date_columns].dtypes)

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print("Total Missing Values:", df.isnull().sum().sum())

    missing = df.isnull().sum()
    print(missing[missing > 0])

In [ ]:
for name, df in datasets.items():
    print(name, ":", df.duplicated().sum(), "duplicate rows")

In [ ]:
for name, df in datasets.items():
    print(name, ":", df.duplicated().sum(), "duplicate rows")

In [ ]:
id_columns = {
    'Customers': (customers, 'customer_id'),
    'Orders': (orders, 'order_id'),
    'Products': (products, 'product_id'),
    'Sellers': (sellers, 'seller_id'),
    'Reviews': (reviews, 'review_id')
}

for name, (df, col) in id_columns.items():
    print(
        name,
        "Total IDs:", df[col].count(),
        "| Unique IDs:", df[col].nunique(),
        "| Duplicate IDs:", df[col].duplicated().sum()
    )

In [ ]:
duplicate_reviews = reviews[
    reviews['review_id'].duplicated(keep=False)
]

print("Total rows with repeated Review IDs:",
      len(duplicate_reviews))

print("\nSample duplicate records:")
display(duplicate_reviews.head(10))

In [ ]:
review_id_check = (
    reviews.groupby('review_id')
    .agg(
        Total_Records=('order_id', 'count'),
        Unique_Orders=('order_id', 'nunique'),
        Unique_Scores=('review_score', 'nunique')
    )
)

print("Review IDs linked to multiple orders:")
display(review_id_check[review_id_check['Unique_Orders'] > 1].head(10))

print("Total Review IDs linked to multiple orders:",
      (review_id_check['Unique_Orders'] > 1).sum())

In [ ]:
print("ORDER ITEMS")
print(order_items[['price', 'freight_value']].dtypes)

print("\nPAYMENTS")
print(payments[['payment_value', 'payment_installments']].dtypes)

print("\nREVIEWS")
print(reviews[['review_score']].dtypes)

In [ ]:
print("ORDER ITEMS")
print("Negative prices:", (order_items['price'] < 0).sum())
print("Negative freight values:", (order_items['freight_value'] < 0).sum())

print("\nPAYMENTS")
print("Negative payment values:", (payments['payment_value'] < 0).sum())
print("Zero payment values:", (payments['payment_value'] == 0).sum())

print("\nREVIEWS")
print("Invalid review scores:",
      (~reviews['review_score'].between(1, 5)).sum())

In [ ]:
zero_payments = payments[payments['payment_value'] == 0]

print("Number of zero-payment records:", len(zero_payments))

display(zero_payments)

In [ ]:
categorical_columns = {
    'Orders': (orders, ['order_status']),
    'Order Items': (order_items, ['seller_id']),
    'Payments': (payments, ['payment_type']),
    'Reviews': (reviews, ['review_score'])
}

for name, (df, columns) in categorical_columns.items():
    print("\n", name)
    for col in columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))

In [ ]:
for name, df in datasets.items():
    date_columns = [col for col in df.columns
                    if 'date' in col.lower() or 'time' in col.lower()]

    print("\n", name)
    for col in date_columns:
        print(col, ":", df[col].dtype)

In [ ]:
print("Delivered before purchase:",
      (orders['order_delivered_customer_date'] <
       orders['order_purchase_timestamp']).sum())

print("Carrier delivery before purchase:",
      (orders['order_delivered_carrier_date'] <
       orders['order_purchase_timestamp']).sum())

print("Customer delivery after estimated date:",
      (orders['order_delivered_customer_date'] >
       orders['order_estimated_delivery_date']).sum())

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print("Total Missing Values:", df.isnull().sum().sum())
    print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print(orders.groupby('order_status')[
    ['order_approved_at',
     'order_delivered_carrier_date',
     'order_delivered_customer_date',
     'Delivery_days']
].apply(lambda x: x.isnull().sum()))

In [ ]:
delivered_missing = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
]

display(delivered_missing)

In [ ]:
datasets = {
    'Customers': customers,
    'Orders': orders,
    'Geolocation': geolocation,
    'Order Items': order_items,
    'Product Category': product_category,
    'Sellers': sellers,
    'Products': products,
    'Reviews': reviews,
    'Payments': payments
}

for name, df in datasets.items():
    print(name, ":", df.duplicated().sum(), "duplicate rows")

In [ ]:
print("Customers:", customers['customer_id'].duplicated().sum())
print("Orders:", orders['order_id'].duplicated().sum())
print("Sellers:", sellers['seller_id'].duplicated().sum())
print("Products:", products['product_id'].duplicated().sum())

In [ ]:
print("Negative Product Prices:",
      (order_items['price'] < 0).sum())

print("Negative Freight Values:",
      (order_items['freight_value'] < 0).sum())

print("Negative Payment Values:",
      (payments['payment_value'] < 0).sum())

In [ ]:
print(orders.groupby('order_status')[
    ['order_approved_at',
     'order_delivered_carrier_date',
     'order_delivered_customer_date',
     'Delivery_days']
].apply(lambda x: x.isnull().sum()))

In [ ]:
display(delivered_missing[[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'Delivery_days'
]])

In [ ]:
print("Missing customer delivery dates:",
      orders['order_delivered_customer_date'].isnull().sum())

print("Missing Delivery_days:",
      orders['Delivery_days'].isnull().sum())

In [ ]:
print(orders.groupby('order_status')['Delivery_days']
      .apply(lambda x: x.isnull().sum()))

In [ ]:
products[products.isnull().any(axis=1)].head(10)

In [ ]:
print("Missing Values in Products Dataset:")
print(products.isnull().sum())

In [ ]:
missing_products = products[
    products['product_category_name'].isnull()
]

print("Products with missing category:", len(missing_products))

print("\nMissing values in these products:")
print(missing_products.isnull().sum())

In [ ]:
products['product_cat_clean'] = products['product_cat_clean'].fillna('unknown')

print("Remaining missing values:",
      products['product_cat_clean'].isnull().sum())

In [ ]:
print("Duplicate Product IDs:",
      products['product_id'].duplicated().sum())

In [ ]:
print("Duplicate Product Rows:", products.duplicated().sum())

In [ ]:
columns = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in columns:
    print(col, ":", (products[col] <= 0).sum())

In [ ]:
display(products.loc[
    products['product_weight_g'] <= 0,
    ['product_id', 'product_weight_g']
])

In [ ]:
display(products.loc[
    products['product_weight_g'] <= 0,
    ['product_id', 'product_cat_clean',
     'product_weight_g', 'product_length_cm',
     'product_height_cm', 'product_width_cm']
])

In [ ]:
category_weights = products.loc[
    (products['product_cat_clean'] == 'cama_mesa_banho') &
    (products['product_weight_g'] > 0),
    'product_weight_g'
]

print("Valid products in this category:", len(category_weights))
print("Average weight:", category_weights.mean())
print("Median weight:", category_weights.median())

In [ ]:
products.loc[
    products['product_weight_g'] <= 0,
    'product_weight_g'
] = category_weights.median()

print("Remaining invalid weights:",
      (products['product_weight_g'] <= 0).sum())

In [ ]:
print("Missing Values:")
print(products.isnull().sum())

print("\nDuplicate Rows:",
      products.duplicated().sum())

print("\nDataset Shape:", products.shape)

In [ ]:
  display(products[products.isnull().any(axis=1)])

In [ ]:
display(products[
    (products['product_cat_clean'] != 'unknown') &
    (products.isnull().any(axis=1))
])

In [ ]:
print("Final Products Shape:", products.shape)
print("\nFinal Missing Values:")
print(products.isnull().sum())

In [ ]:
sellers.head()

In [ ]:
print("Dataset Shape:", sellers.shape)

print("\nDataset Information:")
sellers.info()

print("\nMissing Values:")
print(sellers.isnull().sum())

In [ ]:
print("Duplicate Seller IDs:", sellers['seller_id'].duplicated().sum())

In [ ]:
geolocation.head()

In [ ]:
print("Dataset Shape:", geolocation.shape)

print("\nMissing Values:")
print(geolocation.isnull().sum())

In [ ]:
print("Duplicate Rows:", geolocation.duplicated().sum())

# KPI 1

In [ ]:
total_revenue = payments['payment_value'].sum()

print("Total Revenue: ₹", round(total_revenue, 2))

# KPI 2

In [ ]:
total_orders = orders['order_id'].nunique()

print("Total Orders:", total_orders)

# KPI 3

In [ ]:
# Identify Pet Shop products
pet_products = products[
    products['product_cat_clean'] == 'pet_shop'
]

# Find orders containing Pet Shop products
pet_orders = order_items[
    order_items['product_id'].isin(pet_products['product_id'])
]

# Get delivery dates for those orders
pet_delivery = pet_orders[['order_id']].drop_duplicates().merge(
    orders[['order_id', 'order_purchase_timestamp',
            'order_delivered_customer_date']],
    on='order_id',
    how='left'
)

# Calculate delivery time in days
pet_delivery['delivery_days'] = (
    pd.to_datetime(pet_delivery['order_delivered_customer_date']) -
    pd.to_datetime(pet_delivery['order_purchase_timestamp'])
).dt.days

# Calculate average delivery time
average_pet_delivery = pet_delivery['delivery_days'].dropna().mean()

print("Average Pet Shop Delivery Time:",
      round(average_pet_delivery, 2), "days")

# KPI 4

In [ ]:
# Identify customers from Sao Paulo
import matplotlib.pyplot as plt
sao_paulo_customers = customers[
    customers['customer_city'] == 'sao paulo'
]

# Get their orders
sao_paulo_orders = sao_paulo_customers.merge(
    orders,
    on='customer_id',
    how='inner'
)

# Calculate average product price
sao_paulo_items = sao_paulo_orders[['order_id']].merge(
    order_items[['order_id', 'price']],
    on='order_id',
    how='inner'
)

average_price = sao_paulo_items['price'].mean()

# Calculate average payment value
sao_paulo_payments = sao_paulo_orders[['order_id']].merge(
    payments[['order_id', 'payment_value']],
    on='order_id',
    how='inner'
)

average_payment = sao_paulo_payments['payment_value'].mean()

# Display KPI results
print("KPI 4 - Sao Paulo Analysis")
print("Average Product Price:", round(average_price, 2))
print("Average Payment Value:", round(average_payment, 2))

# Create bar graph
plt.figure(figsize=(8, 5))

labels = ['Average Product Price', 'Average Payment Value']
values = [average_price, average_payment]

bars = plt.bar(labels, values)

plt.title('Average Product Price vs Payment Value - Sao Paulo')
plt.ylabel('Average Value')
plt.xlabel('Category')

# Display values above bars
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        round(bar.get_height(), 2),
        ha='center',
        va='bottom'
    )

plt.tight_layout()
plt.show()

# KPI 5

In [ ]:
# Calculate shipping days
delivery_data = orders[
    ['order_id', 'order_purchase_timestamp',
     'order_delivered_customer_date']
].copy()

delivery_data['delivery_days'] = (
    pd.to_datetime(delivery_data['order_delivered_customer_date']) -
    pd.to_datetime(delivery_data['order_purchase_timestamp'])
).dt.days

# Merge delivery data with reviews
review_delivery = reviews.merge(
    delivery_data[['order_id', 'delivery_days']],
    on='order_id',
    how='inner'
)

# Average shipping days by review score
shipping_review = review_delivery.groupby(
    'review_score'
)['delivery_days'].mean()

print("Average Shipping Days by Review Score:")
print(shipping_review.round(2))

# Create bar graph
plt.figure(figsize=(9, 5))

bars = plt.bar(
    shipping_review.index.astype(str),
    shipping_review.values
)

plt.title('Average Shipping Days vs Review Score')
plt.xlabel('Review Score')
plt.ylabel('Average Shipping Days')
plt.xticks([1, 2, 3, 4, 5])

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        round(bar.get_height(), 2),
        ha='center',
        va='bottom'
    )

plt.tight_layout()
plt.show()

# KPI 6

In [ ]:
import matplotlib.pyplot as plt

total_product_sales = order_items['price'].sum()

print("Total Product Sales: ₹", round(total_product_sales, 2))

# Bar graph
plt.figure(figsize=(7, 5))
plt.bar(['Total Product Sales'], [total_product_sales])

plt.title('Total Product Sales')
plt.ylabel('Sales Value (₹)')
plt.ticklabel_format(axis='y', style='plain')
plt.tight_layout()
plt.show()

# KPI 7

In [ ]:
import matplotlib.pyplot as plt

# Merge order items with product details
category_sales = order_items.merge(
    products[['product_id', 'product_cat_clean']],
    on='product_id',
    how='left'
)

# Calculate sales for each product category
category_sales = category_sales.groupby(
    'product_cat_clean'
)['price'].sum().sort_values(ascending=False)

# Display top 10 categories
top_10_categories = category_sales.head(10)

print("Top 10 Product Categories by Total Sales:")
print(top_10_categories)

# Create graph
plt.figure(figsize=(10, 6))
plt.barh(top_10_categories.index, top_10_categories.values)

plt.title('Top 10 Product Categories by Total Sales')
plt.xlabel('Total Sales Value (₹)')
plt.ylabel('Product Category')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# KPI 8

In [ ]:
import matplotlib.pyplot as plt

# Get customer state and order details
state_orders = customers[['customer_id', 'customer_state']].merge(
    orders[['customer_id', 'order_id']],
    on='customer_id',
    how='inner'
)

# Calculate total payment for each order
order_payments = payments.groupby('order_id')['payment_value'].sum().reset_index()

# Merge payment details
state_orders = state_orders.merge(
    order_payments,
    on='order_id',
    how='inner'
)

# Calculate average order value by state
state_aov = state_orders.groupby(
    'customer_state'
)['payment_value'].mean().sort_values(ascending=False)

print("Average Order Value by Customer State:")
print(state_aov)

# Create graph
plt.figure(figsize=(12, 6))
plt.bar(state_aov.index, state_aov.values)

plt.title('Average Order Value by Customer State')
plt.xlabel('Customer State')
plt.ylabel('Average Order Value (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# KPI 9

In [ ]:
import matplotlib.pyplot as plt

# Merge order items with seller details
seller_orders = order_items.merge(
    sellers[['seller_id', 'seller_state']],
    on='seller_id',
    how='left'
)

# Count unique orders by seller state
state_order_count = seller_orders.groupby(
    'seller_state'
)['order_id'].nunique().sort_values(ascending=False)

# Display top 10 states
top_10_states = state_order_count.head(10)

print("Top 10 Seller States by Number of Orders:")
print(top_10_states)

# Create graph
plt.figure(figsize=(10, 6))
plt.bar(top_10_states.index, top_10_states.values)

plt.title('Top 10 Seller States by Number of Orders')
plt.xlabel('Seller State')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

# KPI 10

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert purchase date into datetime format
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

# Extract year and month
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

# Count unique orders for each month
monthly_orders = orders.groupby('year_month')['order_id'].nunique()

# Display results
print("Month-over-Month Order Trend:")
print(monthly_orders)

# Create graph
plt.figure(figsize=(14, 6))
plt.plot(
    monthly_orders.index.astype(str),
    monthly_orders.values,
    marker='o'
)

plt.title('Month-over-Month Order Trend')
plt.xlabel('Month')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

# KPI 11

In [ ]:
total_customers = customers['customer_id'].nunique()

print("Total Number of Customers:", total_customers)

# KPI 12

In [ ]:
total_revenue = payments['payment_value'].sum()

print("Total Revenue: ₹", round(total_revenue, 2))

# KPI 13

In [ ]:
total_product_price = order_items['price'].sum()

print("Total Product Price: ₹", round(total_product_price, 2))